In [1]:
import pandas as pd
import glob
import os
import re

In [3]:
nanopore_index = pd.read_excel("../data_file/SPN_nanopore_Nader/Combined_data_for_SPN_nanopore_WGS.xlsx")
map_table = pd.read_excel("../data_file/Look_up_table.xlsx")

nanopore_filename = pd.read_csv("../data_file/barcode.txt", sep="\t", header=None, names=["nanopore_filename"])

In [5]:
nanopore_index

,Genome Name,Genome Length,No. Contigs,Smallest Contig,Largest Contig,Average Contig Length,N50,GC Content,Genome fraction (%),Lineage,...,amoxmic_ear,methomic_pre_ear,methomic_ear,levomic_pre_ear,levomic_ear,dcollect_ear,pcn_sens_ear,accessionNUM_ear,LDX,RDX
0,01_barcode01.contigs,2104635.0,2.0,25169.0,2079466.0,1052317.0,2079466.0,39.7,85.751,3,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,AOM,AOM
1,01_barcode03.contigs,2289937.0,6.0,13810.0,1705204.0,381656.0,1705204.0,39.7,88.722,11,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,No effusion,No effusion
2,01_barcode05.contigs,2058639.0,2.0,12414.0,2046225.0,1029319.0,2046225.0,39.8,85.137,59,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,AOM,No effusion
3,01_barcode09.contigs,2100474.0,2.0,10602.0,2089872.0,1050237.0,2089872.0,39.6,90.040,5,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,No effusion,No effusion
4,01_barcode10.contigs,2081348.0,1.0,2081348.0,2081348.0,2081348.0,2081348.0,39.6,86.318,7,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,No effusion,No effusion
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,04_barcode86.contigs,2160830.0,8.0,13053.0,922353.0,270103.0,513729.0,39.7,87.318,36,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,AOM,No effusion
123,04_barcode94.contigs,2136584.0,26.0,3102.0,548120.0,82176.0,219604.0,39.6,87.196,Not assigned,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,AOM,NaN
124,04_barcode95.contigs,2244725.0,15.0,6288.0,707650.0,149648.0,292058.0,39.8,90.291,5,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,OME,AOM
125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN


In [7]:
nanopore_filename

,nanopore_filename
0,Batch1_4.barcode12_33.fastq.gz
1,Batch1.barcode01.fastq.gz
2,Batch1.barcode02.fastq.gz
3,Batch1.barcode03.fastq.gz
4,Batch1.barcode04.fastq.gz
...,...
216,Batch4.barcode86.fastq.gz
217,Batch4.barcode87.fastq.gz
218,Batch4.barcode89.fastq.gz
219,Batch4.barcode94.fastq.gz


In [9]:
# rename column from nanopore metadata file
nanopore_index = nanopore_index.rename(columns={'accessionNum': 'Accession No', "Genome Name": "Nanopore_index"})
# rename column from illumina metadata file
map_table = map_table.rename(columns={"Our number":"Illumina_index"})

# extract batch and barcode numbers
nanopore_index[['batch', 'barcode']] = nanopore_index['Nanopore_index'].str.extract(r'(\d)_barcode(\d+)')
# build a column that matches nanopore data file name
nanopore_index['fastq_name'] = (
    'Batch' + nanopore_index['batch'] +
    '.barcode' + nanopore_index['barcode'] +
    '.fastq.gz'
)

target1 = map_table[["Accession No", "Illumina_index"]]
target2 = nanopore_index[["Nanopore_index", "fastq_name", "Accession No"]]

In [13]:
nanopore_index[nanopore_index["fastq_name"] == "Batch2.barcode09.fastq.gz"]

,Nanopore_index,Genome Length,No. Contigs,Smallest Contig,Largest Contig,Average Contig Length,N50,GC Content,Genome fraction (%),Lineage,...,levomic_pre_ear,levomic_ear,dcollect_ear,pcn_sens_ear,accessionNUM_ear,LDX,RDX,batch,barcode,fastq_name
35,02_barcode09.contigs,2144936.0,4.0,12937.0,1290325.0,536234.0,1290325.0,39.6,86.576,6,...,=,1.0,2023-03-31,Intermediate,F18448840,AOM,AOM,2,09,Batch2.barcode09.fastq.gz


In [15]:
nanopore_index["Accession No"]
nanopore_index[nanopore_index["Accession No"] == "F18448840"]

,Nanopore_index,Genome Length,No. Contigs,Smallest Contig,Largest Contig,Average Contig Length,N50,GC Content,Genome fraction (%),Lineage,...,levomic_pre_ear,levomic_ear,dcollect_ear,pcn_sens_ear,accessionNUM_ear,LDX,RDX,batch,barcode,fastq_name
35,02_barcode09.contigs,2144936.0,4.0,12937.0,1290325.0,536234.0,1290325.0,39.6,86.576,6,...,=,1.0,2023-03-31,Intermediate,F18448840,AOM,AOM,2,09,Batch2.barcode09.fastq.gz


In [17]:
target2

,Nanopore_index,fastq_name,Accession No
0,01_barcode01.contigs,Batch1.barcode01.fastq.gz,W16709299
1,01_barcode03.contigs,Batch1.barcode03.fastq.gz,W17143487
2,01_barcode05.contigs,Batch1.barcode05.fastq.gz,F16380623
3,01_barcode09.contigs,Batch1.barcode09.fastq.gz,T17601862
4,01_barcode10.contigs,Batch1.barcode10.fastq.gz,F16721574
...,...,...,...
122,04_barcode86.contigs,Batch4.barcode86.fastq.gz,T20596539
123,04_barcode94.contigs,Batch4.barcode94.fastq.gz,W20470375
124,04_barcode95.contigs,Batch4.barcode95.fastq.gz,H20331969
125,NaN,NaN,NaN


In [19]:
merged1 = pd.merge(
    target1,
    target2,
    left_on='Accession No',
    right_on='Accession No',
    how='inner'
)

In [21]:
merged1

,Accession No,Illumina_index,Nanopore_index,fastq_name
0,W15810037,1,02_barcode52.contigs,Batch2.barcode52.fastq.gz
1,W17143487,2,01_barcode03.contigs,Batch1.barcode03.fastq.gz
2,F16380623,3,01_barcode05.contigs,Batch1.barcode05.fastq.gz
3,W17653277,4,01_barcode11.contigs,Batch1.barcode11.fastq.gz
4,M17701635,5,01_barcode12.contigs,Batch1.barcode12.fastq.gz
5,M17701635,5,04_barcode33.contigs,Batch4.barcode33.fastq.gz
6,T18271326,6,02_barcode90.contigs,Batch2.barcode90.fastq.gz
7,M18434558,7,01_barcode34.contigs,Batch1.barcode34.fastq.gz
8,F17654457,8,01_barcode35.contigs,Batch1.barcode35.fastq.gz
9,F17896493,9,03_barcode89.contigs,Batch3.barcode89.fastq.gz


In [23]:
merged2 = pd.merge(
    merged1,
    nanopore_filename,
    left_on='fastq_name',
    right_on='nanopore_filename',
    how='left'
)

In [25]:
merged2

,Accession No,Illumina_index,Nanopore_index,fastq_name,nanopore_filename
0,W15810037,1,02_barcode52.contigs,Batch2.barcode52.fastq.gz,Batch2.barcode52.fastq.gz
1,W17143487,2,01_barcode03.contigs,Batch1.barcode03.fastq.gz,Batch1.barcode03.fastq.gz
2,F16380623,3,01_barcode05.contigs,Batch1.barcode05.fastq.gz,Batch1.barcode05.fastq.gz
3,W17653277,4,01_barcode11.contigs,Batch1.barcode11.fastq.gz,Batch1.barcode11.fastq.gz
4,M17701635,5,01_barcode12.contigs,Batch1.barcode12.fastq.gz,NaN
5,M17701635,5,04_barcode33.contigs,Batch4.barcode33.fastq.gz,NaN
6,T18271326,6,02_barcode90.contigs,Batch2.barcode90.fastq.gz,Batch2.barcode90.fastq.gz
7,M18434558,7,01_barcode34.contigs,Batch1.barcode34.fastq.gz,Batch1.barcode34.fastq.gz
8,F17654457,8,01_barcode35.contigs,Batch1.barcode35.fastq.gz,Batch1.barcode35.fastq.gz
9,F17896493,9,03_barcode89.contigs,Batch3.barcode89.fastq.gz,Batch3.barcode89.fastq.gz


In [27]:
merged2["illumina"] = (
    "i"
    + merged2["Illumina_index"].astype(str)
    + "_"
    + merged2["Nanopore_index"].str.replace(".contigs", "", regex=False)
    + "_illumina.fastq.gz"
)

merged2["nanopore"] = (
    "i"
    + merged2["Illumina_index"].astype(str)
    + "_"
    + merged2["Nanopore_index"].str.replace(".contigs", "", regex=False)
    + "_nanopore.fastq.gz"
)


In [29]:
merged2

,Accession No,Illumina_index,Nanopore_index,fastq_name,nanopore_filename,illumina,nanopore
0,W15810037,1,02_barcode52.contigs,Batch2.barcode52.fastq.gz,Batch2.barcode52.fastq.gz,i1_02_barcode52_illumina.fastq.gz,i1_02_barcode52_nanopore.fastq.gz
1,W17143487,2,01_barcode03.contigs,Batch1.barcode03.fastq.gz,Batch1.barcode03.fastq.gz,i2_01_barcode03_illumina.fastq.gz,i2_01_barcode03_nanopore.fastq.gz
2,F16380623,3,01_barcode05.contigs,Batch1.barcode05.fastq.gz,Batch1.barcode05.fastq.gz,i3_01_barcode05_illumina.fastq.gz,i3_01_barcode05_nanopore.fastq.gz
3,W17653277,4,01_barcode11.contigs,Batch1.barcode11.fastq.gz,Batch1.barcode11.fastq.gz,i4_01_barcode11_illumina.fastq.gz,i4_01_barcode11_nanopore.fastq.gz
4,M17701635,5,01_barcode12.contigs,Batch1.barcode12.fastq.gz,NaN,i5_01_barcode12_illumina.fastq.gz,i5_01_barcode12_nanopore.fastq.gz
5,M17701635,5,04_barcode33.contigs,Batch4.barcode33.fastq.gz,NaN,i5_04_barcode33_illumina.fastq.gz,i5_04_barcode33_nanopore.fastq.gz
6,T18271326,6,02_barcode90.contigs,Batch2.barcode90.fastq.gz,Batch2.barcode90.fastq.gz,i6_02_barcode90_illumina.fastq.gz,i6_02_barcode90_nanopore.fastq.gz
7,M18434558,7,01_barcode34.contigs,Batch1.barcode34.fastq.gz,Batch1.barcode34.fastq.gz,i7_01_barcode34_illumina.fastq.gz,i7_01_barcode34_nanopore.fastq.gz
8,F17654457,8,01_barcode35.contigs,Batch1.barcode35.fastq.gz,Batch1.barcode35.fastq.gz,i8_01_barcode35_illumina.fastq.gz,i8_01_barcode35_nanopore.fastq.gz
9,F17896493,9,03_barcode89.contigs,Batch3.barcode89.fastq.gz,Batch3.barcode89.fastq.gz,i9_03_barcode89_illumina.fastq.gz,i9_03_barcode89_nanopore.fastq.gz


In [31]:
merged2.to_csv("../data_file/illumina_nanopore_mapping.csv", index=False)

In [35]:
merged2.to_excel("../data_file/illumina_nanopore_mapping.xlsx", index=False)

In [ ]:
# illumina: ix_nx_bx_illumina.fastq.gz
# ex: Batch2.barcode90.fastq.gz with illumina index 6 -> i6_n2_b90_illumina.fastq.gz
# nanopore: ix_nx_bx_nanopore.fastq.gz
# ex: i6_n2_b90_nanopore.fastq.gz